In [ ]:
# This script performs a PCA followed by a k-mean clustering per country and create a polygon layer of 
# clusters and specific attributes
import arcpy
import pandas as pd
import numpy as np
from collections import defaultdict
import os
import seaborn as sns
import matplotlib.pyplot as plt
import sys
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr
import matplotlib as mpl
mpl.rcParams["legend.fontsize"] = 13

# Add the folder to sys.path
script_folder = "..."
if script_folder not in sys.path:
    sys.path.append(script_folder)
from PCA_plain import PC_analysis, make_3D_plot_PCA, make_2D_plot_PCA

arcpy.env.workspace=r'...gdb'
path=r'...'
path_to_data = r'...'

In [ ]:
# import the cleaned dataset where spatial join of census and AQ data and attribution of country code and DEGURBA (2021) 
census_AQ_EU_urban=pd.read_csv(path_to_data+'\\'+'census2021_AQ_EU28_urban.csv')

# country list
countries=np.unique(census_AQ_EU_urban['CNTR_CODE'])
print(countries)

['AT' 'BE' 'BG' 'CY' 'CZ' 'DE' 'DK' 'EE' 'EL' 'ES' 'FI' 'FR' 'HR' 'HU'
 'IE' 'IT' 'LT' 'LU' 'LV' 'MT' 'NL' 'NO' 'PL' 'PT' 'RO' 'SE' 'SI' 'SK']


In [ ]:
# EU- versus country-level analyses
# create a scatter plot for each country 
n_cols = 4  # columns in the subplot grid
n_rows = len(countries) // n_cols + (len(countries) % n_cols > 0)  # number of rows needed

for pollutant in ['pm_10','pm_25','no_2', 'o3_p932']:
    for var in [ 'EU_OTH_share','EMP_share','M_share', 'F_share', 'Y_LT15_share', 'Y_1564_share', 'Y_GE65_share', 'NAT_share', 'OTH_share']: #
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5*n_rows))
        axes = axes.flatten()  
        custom_palette = ["#FF5733", "#33FF57", "#3357FF"] 
        
        # country-wise scatter plots
        for ax, country in zip(axes, countries):
            country_data = census_AQ_EU_urban[census_AQ_EU_urban['CNTR_CODE'] == country] # filter data for the current country
        
            # Spearman correlation coefficient
            corr21, _21 = spearmanr(country_data[pollutant].loc[country_data['RASTERVALU']==21], country_data[var].loc[country_data['RASTERVALU']==21])
            corr22, _22 = spearmanr(country_data[pollutant].loc[country_data['RASTERVALU']==22], country_data[var].loc[country_data['RASTERVALU']==22])
            corr30, _30 = spearmanr(country_data[pollutant].loc[country_data['RASTERVALU']==30], country_data[var].loc[country_data['RASTERVALU']==30])
            corr,_ = spearmanr(country_data[pollutant].values, country_data[var].values)
            
            # scatter plot
            sns.scatterplot(data=country_data, x=pollutant, y=var, ax=ax, hue='RASTERVALU', palette=custom_palette) #
            ax.set_title(f'{country}\n \nSpearman corr: 21:{corr21:.2f}, 22:{corr22:.2f}, 30:{corr30:.2f}, all:{corr:.2f}')
            # ax.set_title(f'{country}\n \nSpearman corr: {corr:.2f}')
            ax.set_xlabel(pollutant,fontsize=12)
            ax.set_ylabel(var,fontsize=12)
        
        plt.tight_layout()
        plt.savefig(path_to_data+'\\'+'country_wise_spearman_scatter_%s_%s.png'%(pollutant, var), dpi=500)
        
        # single scatter plot for all countries
        plt.figure(figsize=(12, 8))
        scatter = sns.scatterplot(data=census_AQ_EU_urban, x=pollutant, y=var, hue='RASTERVALU', palette=custom_palette) #, hue='RASTERVALU'
        
        # Spearman correlation coefficient for the entire dataset
        corr21, _21 = spearmanr(census_AQ_EU_urban[pollutant].loc[census_AQ_EU_urban['RASTERVALU']==21], census_AQ_EU_urban[var].loc[census_AQ_EU_urban['RASTERVALU']==21])
        corr22, _22 = spearmanr(census_AQ_EU_urban[pollutant].loc[census_AQ_EU_urban['RASTERVALU']==22], census_AQ_EU_urban[var].loc[census_AQ_EU_urban['RASTERVALU']==22])
        corr30, _30 = spearmanr(census_AQ_EU_urban[pollutant].loc[census_AQ_EU_urban['RASTERVALU']==30], census_AQ_EU_urban[var].loc[census_AQ_EU_urban['RASTERVALU']==30])
        corr,_ = spearmanr(census_AQ_EU_urban[pollutant].values, census_AQ_EU_urban[var].values)
        
        plt.title(f'Spearman corr: 21:{corr21:.2f}, 22:{corr22:.2f}, 30:{corr30:.2f}, all:{corr:.2f}')
        plt.xlabel(pollutant, fontsize=12)
        plt.ylabel(var, fontsize=12)
        plt.tight_layout()
        plt.savefig(path_to_data+'\\'+'EU_wide_spearman_scatter_%s_%s.png'%(pollutant, var), dpi=500)


In [3]:
# Stats and Spearman correlations between variables
census_AQ_EU_urban.describe().to_csv(path_to_data+'\\'+'EU_wide_urban_summary.csv')
census_AQ_EU_urban[['pm_25',
       'pm_10', 'no_2', 'o3_s35', 'o3_p932', 'RASTERVALU',
       'T', 'M', 'F', 'Y_LT15', 'Y_1564', 'Y_GE65',
       'EMP', 'NAT', 'EU_OTH', 'OTH', 'SAME', 'CHG_IN', 'CHG_OUT',
       'M_F_total', 'Y_LT15_Y_1564_Y_GE65_total', 'NAT_EU_OTH_OTH_total',
       'SAME_CHG_IN_CHG_OUT_total', 'M_share', 'F_share', 'Y_LT15_share',
       'Y_1564_share', 'Y_GE65_share', 'NAT_share', 'EU_OTH_share',
       'OTH_share', 'SAME_share', 'CHG_IN_share', 'CHG_OUT_share', 'EMP_share',
       'urban']].corr(method='spearman').to_csv(path_to_data+'\\'+'EU_wide_urban_corr.csv')

In [ ]:
# general overview: boxplots of AQ and census data per deg of urbanisation
for what in ['Y_1564_share', 'M_share','F_share','Y_LT15_share','Y_GE65_share', 'NAT_share',
             'OTH_share','EU_OTH_share',, 'SAME_share', 'CHG_IN_share','CHG_OUT_share', 'EMP_share']:   
    plt.figure(figsize=(10, 8))
    sns.boxplot(y=what, x="RASTERVALU", #y=what,hue="RASTERVALU"
                 legend=False, palette=["mediumaquamarine", 'teal', 'lightblue'],
                data=census_AQ_EU_urban)
    sns.despine(offset=5, trim=True)
    plt.xlabel('Degree of urbanisation', fontsize=13)
    plt.ylabel('%', fontsize=13)
    plt.tight_layout()
    plt.savefig(path_to_data+'\\4 Census categories by DEGURBA\\'+"census2021_box_%s_urban_degrees_all_urban.png"%(what),
                dpi=500) 

    plt.figure(figsize=(15, 8))
    sns.boxplot(x='CNTR_CODE', y=what,
                 hue="RASTERVALU",palette=["mediumaquamarine", 'teal', 'lightblue'], 
                data=census_AQ_EU_urban.sort_values(what), flierprops={"marker": "."})
    sns.despine(offset=10, trim=True)
    plt.xlabel('Country', fontsize=13)
    plt.ylabel('%', fontsize=13)#'Degree of urbanisation'
    # plt.legend(title='', fontsize=12)
    plt.tight_layout()
    plt.savefig(path_to_data+'\\4 Census categories by DEGURBA\\'+"census2021_box_%s_urban_degrees_countries_all_urban.png"%(what),
                dpi=600) 

for pollutant in [ 'pm_10', 'no_2', 'pm_25', 'o3_s35', 'o3_p932']:
    plt.figure(figsize=(8, 8))
    sns.boxplot(x="RASTERVALU", y=pollutant, palette=["mediumaquamarine", 'teal', 'lightblue'],
                data=census_AQ_EU_urban)
    sns.despine(offset=10, trim=True)
    plt.xlabel('Degree of urbanisation', fontsize=12)
    plt.ylabel('ug/m3', fontsize=12)
    plt.tight_layout()
    plt.savefig(path_to_data+'\\3 AQ by DEGURBA\\'+"census2021_box_%s_urban_degrees_all_urban.png"%(pollutant),
                dpi=500) 
    
    plt.figure(figsize=(15, 8))
    sns.boxplot(x='CNTR_CODE', y=pollutant,
                palette=["mediumaquamarine", 'teal', 'lightblue'], hue="RASTERVALU",
                data=census_AQ_EU_urban.sort_values('CNTR_CODE'), flierprops={"marker": "."})
    sns.despine(offset=10, trim=True)
    plt.tight_layout()
    plt.xlabel('Country', fontsize=13)
    plt.ylabel('ug/m3', fontsize=13)#'Degree of urbanisation'
    # plt.legend(title='', fontsize=12)
    plt.savefig(path_to_data+'\\3 AQ by DEGURBA\\'+"census2021_box_%s_urban_degrees_countries_all_urban.png"%(pollutant),
                dpi=600) 


In [ ]:
# run the clustering analyses
country_selection=countries
print(len(country_selection))
batch='' #'_batch3'
# import the original dataset to get coordinates
census=pd.read_csv(path_to_data+'\\'+'census2021v2_1989_country_AQ2021_REDEGURBA_coord.csv') 

In [ ]:
# principal component analysis for each country and k-mean clustering
# save the statistical parameters in a single file for all countries
EU_PCA_clusters_info=pd.DataFrame({})
stats, ks, PCs, countries, features = [], [], [], [], []

for country in country_selection:        
    ## keep only the variables of interest
    census_AQ_EU_urban_cntr=census_AQ_EU_urban.loc[census_AQ_EU_urban['CNTR_CODE']==country]
    data_PCA=census_AQ_EU_urban_cntr[['pm_25',
            'pm_10', 'no_2', 'o3_s35', 'o3_p932', 'RASTERVALU',
            'Y_LT15_share',#'F_share','M_share',
            'Y_1564_share', 'Y_GE65_share', 'NAT_share', 'EU_OTH_share',
            'OTH_share', 'SAME_share', 'CHG_IN_share', 'CHG_OUT_share',
            'EMP_share']].copy()
    data_PCA.index=np.arange(0,len(data_PCA))
    
    ## save the dataset used in the PCA
    main='AQ_census_2021'
    scenario='%s_%s'%(main,country)
    data_PCA.to_csv(path+'\\'+ scenario+'_PCA_inputdataset.csv', index=False)
    data_PCA.describe().to_csv(path+'\\'+ scenario+'_PCA_inputdataset_summmary.csv', index=True)
    data_PCA.corr(method='spearman').to_csv(path+'\\'+ scenario+'_PCA_inputdataset_spearman.csv', index=True)

    ## standardise the data
    std_scaler = StandardScaler(with_std=True) 
    scaled_data_PCA = std_scaler.fit_transform(data_PCA)
    scaled_data_PCA_df=pd.DataFrame(scaled_data_PCA, columns=data_PCA.columns)

    ## PCA for all degrees of urbanisation or separate ones
    projected_census_df, PCA_components, pcs=PC_analysis(scaled_data_PCA, len(scaled_data_PCA_df.columns), scenario, 
                                    scaled_data_PCA_df.columns, path)
    print('PCA - Number of PCs for %s: %d'%(country, pcs))
    
    PC_data_kmeans=projected_census_df[0:pcs].T ## select only the PCs of interest!
    PC_data_kmeans=pd.DataFrame(PC_data_kmeans)
    PC_data_kmeans.columns=['PC%d'%i for i in np.arange(0,pcs)]

    ## add the coordinates to the datapoints
    ### get it from the original dataset instead??
    data_coord=pd.merge(census_AQ_EU_urban_cntr, census, how='left', left_on='pointid', right_on='pointid') 
    data_coord.index=np.arange(0,len(data_coord))

    ## save the projected dataset with coordinates
    PC_data_kmeans = pd.concat([PC_data_kmeans, data_coord[['POINT_X',  'POINT_Y','T_x']]], axis=1)
    PC_data_kmeans = PC_data_kmeans.rename(columns={"T_x": "T"})
    PC_data_kmeans.to_csv(path+'\\'+scenario+'_PCA_%dprojecteddataset_coord.csv'%pcs, index=False)  

    ## create a dataset with projected and non-projected variables together
    compiled_data_PCA=pd.concat([data_PCA, PC_data_kmeans], axis=1)
    compiled_data_PCA.to_csv(path+'\\'+ scenario+'_PCA_%dall_dataset_projected.csv'%pcs, index=False)

    ## table to points for geospatial layer
    arcpy.management.XYTableToPoint(
        in_table=path+'\\'+ scenario+'_PCA_%dall_dataset_projected.csv'%pcs,
        out_feature_class=scenario+'_PCA_%dall_dataset_projected_map'%pcs,
        x_field="POINT_X",
        y_field="POINT_Y",
        z_field=None,
        coordinate_system='PROJCS["ETRS_1989_LAEA",GEOGCS["GCS_ETRS_1989",DATUM["D_ETRS_1989",SPHEROID["GRS_1980",6378137.0,298.257222101]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Azimuthal_Equal_Area"],PARAMETER["False_Easting",4321000.0],PARAMETER["False_Northing",3210000.0],PARAMETER["Central_Meridian",10.0],PARAMETER["Latitude_Of_Origin",52.0],UNIT["Meter",1.0]];-8426600 -9526700 10000;-100000 10000;-100000 10000;0.001;0.001;0.001;IsHighPrecision'
    )
    print('PCA - Spatial layer with raw and projected variables created for %s'%(country))

    ## perform kmean clustering with the PCs and with searching of optimum number of clusters
    fields=''
    for pc in np.arange(0,pcs):
        fields+='PC%d;'%pc
    fields = fields[:-1]
    print('The following fields will be used for the k-mean clustering in %s for %d PCs:'%(country,pcs), fields)
    
    arcpy.stats.MultivariateClustering(
        in_features=scenario+'_PCA_%dall_dataset_projected_map'%pcs,
        output_features=scenario+'_PCA_%dall_dataset_projected_map_C'%pcs,
        analysis_fields=fields,
        clustering_method="K_MEANS",
        initialization_method="OPTIMIZED_SEED_LOCATIONS",
        initialization_field=None,
        number_of_clusters=None,
        output_table="kmean_clusters_opt_%s_PCA_nocoord"%country
    )

    ## transform the cluster dataset to raster and then polygon
    arcpy.conversion.PointToRaster(
        in_features=scenario+'_PCA_%dall_dataset_projected_map_C'%pcs,
        value_field="CLUSTER_ID",
        out_rasterdataset=scenario+'_PCA_%dall_dataset_projected_rast_C'%pcs,
        cell_assignment="MOST_FREQUENT",
        priority_field="NONE",
        cellsize=r"...",
        build_rat="BUILD"
    )
    
    arcpy.conversion.RasterToPolygon(
        in_raster=scenario+'_PCA_%dall_dataset_projected_rast_C'%pcs,
        out_polygon_features=scenario+'_PCA_%dall_dataset_projected_pol_C'%pcs,
        simplify="NO_SIMPLIFY",
        raster_field="Value",
        create_multipart_features="SINGLE_OUTER_PART",
        max_vertices_per_feature=None
    )
    print('K-mean clustering - Clustering with projected variables done and ready in a polygon layer for %s'%(country))

    ## for each cluster value add the stat column of interest
    list_stats=[]
    variables=['T','pm_25', 'pm_10', 'no_2', 'o3_s35', 'o3_p932', 'RASTERVALU', #'M_share','F_share',
                'Y_LT15_share', 'Y_1564_share', 'Y_GE65_share', 'NAT_share', 
                'EU_OTH_share', 'OTH_share', 'SAME_share', 'CHG_IN_share', 
                'CHG_OUT_share', 'EMP_share']
    
    for var in variables:
        arcpy.management.AddField(scenario+'_PCA_%dall_dataset_projected_pol_C'%pcs, 'MEAN_%s'%var, 'Double')
        arcpy.management.AddField(scenario+'_PCA_%dall_dataset_projected_pol_C'%pcs, 'STD_%s'%var, 'Double')
        list_stats.append('MEAN_%s'%var)
        list_stats.append('STD_%s'%var)

    ## calc the average and std of the different variables for each cluster value
    ### join the raw features to the cluster dataset first
    fields='POINT_X;POINT_Y'
    for var in variables:
        fields+=';%s'%var
    arcpy.management.JoinField(
        in_data=scenario+'_PCA_%dall_dataset_projected_map_C'%pcs,
        in_field="OBJECTID",
        join_table=scenario+'_PCA_%dall_dataset_projected_map'%pcs,
        join_field="OBJECTID",
        fields=fields,
        fm_option="NOT_USE_FM",
        field_mapping=None,
        index_join_fields="NO_INDEXES"
    )
    print('K-mean clustering - Raw variables added to the cluster map for %s'%(country))

    stat_fields=''
    for var in variables:
        stat_fields+='%s MEAN;%s STD;'%(var,var)
    stat_fields=stat_fields[:-1]    
    arcpy.analysis.Statistics(
        in_table=scenario+'_PCA_%dall_dataset_projected_map_C'%pcs,
        out_table=scenario+'_PCA_%dall_dataset_projected_map_C_stat'%pcs,
        statistics_fields=stat_fields,
        case_field="CLUSTER_ID",
        concatenation_separator=""
    )

    ## transform the table into a dataframe
    array = arcpy.da.TableToNumPyArray(scenario+'_PCA_%dall_dataset_projected_map_C_stat'%pcs, "*")
    df = pd.DataFrame(array)

    ## save df in the EU-level dataframe: add info on CNTR, PC and cluster numbers
    clusters=len(df['CLUSTER_ID'].values)
    df['Clusters']=[clusters for i in range(clusters)]
    df['PCs']=[pcs for i in range(clusters)]
    df['Country']=[country for i in range(clusters)]
    EU_PCA_clusters_info=pd.concat([EU_PCA_clusters_info, df], axis=0)
    
    ## assign to each polygon in the polygon dataset the average of the corresponding cluster
    list_fields=["gridcode"]+list_stats
    with arcpy.da.UpdateCursor(scenario+'_PCA_%dall_dataset_projected_pol_C'%pcs, list_fields) as cursor:
        for row in cursor:
            cluster = row[0]
            for var in variables:
                coli_df=np.where(df.columns == 'MEAN_%s'%var)[0][0]
                colj_df=np.where(df.columns == 'STD_%s'%var)[0][0]
                coli_list=np.where(np.array(list_fields) == 'MEAN_%s'%var)[0][0]
                colj_list=np.where(np.array(list_fields) == 'STD_%s'%var)[0][0]
    
                row[coli_list] = df[df.columns[coli_df]].values[cluster-1]
                row[colj_list] = df[df.columns[colj_df]].values[cluster-1]
            cursor.updateRow(row)

    print('K-mean clustering - Polygon layer with average and std of each AQ and census variable ready for %s'%(country))

    ## show distributions of each variable within the clusters
    ### export attribute table first
    arcpy.conversion.ExportTable(
    in_table=scenario+'_PCA_%dall_dataset_projected_map_C'%pcs,
    out_table=scenario+'_PCA_%dall_dataset_projected_map_C'%pcs+"_Table", # country data with cluster assignment
    where_clause="",
    use_field_alias_as_name="NOT_USE_ALIAS",
    sort_field=None)
    
    array = arcpy.da.TableToNumPyArray(scenario+'_PCA_%dall_dataset_projected_map_C'%pcs+"_Table", "*")
    data = pd.DataFrame(array)
    data.to_csv(path+'\\'+ scenario+'_PCA_%dall_dataset_projected_%dclusters.csv'%(pcs, clusters), index=False)
    
    my_palette=sns.color_palette("tab10", n_colors=10)
    for what in variables:
        if not what == 'T': 
            fig = plt.figure(figsize = (8, 8))
            ax=sns.histplot(data=data, x=what, hue="CLUSTER_ID", kde=True, log_scale=False, palette=my_palette)
            plt.xlabel(what, fontsize=12)
            plt.ylabel('Probability density',fontsize=12)
            plt.legend(fontsize=12)
            plt.tight_layout()
            # plt.show()
            plt.savefig(path+'\\'+scenario+'_PCA_%d_hist_%dclusters_%s.png'%(pcs,clusters,what), dpi=500) 

    var_of_interest=['RASTERVALU', 'Y_LT15_share', 'Y_1564_share', 'Y_GE65_share',
            'NAT_share', 'EU_OTH_share', 'OTH_share', 'EMP_share','CLUSTER_ID']
    df_melted = pd.melt(data[var_of_interest], id_vars=['CLUSTER_ID'], var_name='Variable', value_name='Value')  
    
    ### show boxplots of variable distributions for each cluster
    plt.figure(figsize=(8, 6))
    my_palette=sns.color_palette("tab10", n_colors=clusters)
    sns.boxplot(x='Variable', y='Value', hue='CLUSTER_ID', data=df_melted, palette=my_palette)
    plt.xlabel('Variable', fontsize=12)
    plt.xticks(rotation=45)
    plt.ylabel('Value', fontsize=12)
    # plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(path+'\\'+scenario+'_PCA_%d_box_%dclusters_census_var.png'%(pcs,cluster), dpi=500) 

    df_melted = pd.melt(data[['pm_25','pm_10', 'no_2','o3_p932','CLUSTER_ID']], id_vars=['CLUSTER_ID'], var_name='Variable', value_name='Value')
    ### show boxplots of variable distributions for each cluster
    plt.figure(figsize=(8, 6))
    my_palette=sns.color_palette("tab10", n_colors=clusters)
    sns.boxplot(x='Variable', y='Value', hue='CLUSTER_ID', data=df_melted, palette=my_palette)
    plt.xlabel('Variable', fontsize=12)
    plt.xticks(rotation=45)
    plt.ylabel('Value', fontsize=12)
    # plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(path+'\\'+scenario+'_PCA_%d_box_%dclusters_pollutant_var.png'%(pcs,cluster), dpi=500) 

    # retrieve the clustering parameters and metrics
    array = arcpy.da.TableToNumPyArray("kmean_clusters_opt_%s_PCA_nocoord"%country, "*")
    data = pd.DataFrame(array)
    PCs.append(pcs)
    max_row = data.loc[data['PSEUDO_F'].idxmax()]
    ks.append(max_row['NUM_GROUPS'])
    stats.append(max_row['PSEUDO_F'])
    countries.append(country)

EU_PCA_clusters_info.to_csv(path+'\\'+'%s_EU_PCA_clusters%s.csv'%(main,batch), index=False)
all_stats_ks=pd.DataFrame({
'CNTR_CODE': countries,
'Variables':features,
'PCs':PCs,
'ks': ks,
'F-stat': stats})
all_stats_ks.to_csv(path+'\\'+ '%s_EU_clusters_F_stat.csv'%main)
all_stats_ks.describe().to_csv(path+'\\'+ '%s_EU_clusters_F_stat_stats.csv'%main)